# Cold diffusion

Thin wrapper around [`train.py`](train.py) and [`generate.py`](generate.py): loads [`configs/____.yaml`], trains with `___Degradation`, runs generation, then shows saved sample grids inline.

**Runtime:** use a **GPU** runtime on Colab (or CUDA locally) so training finishes in reasonable time.

## 1. Setup (Colab + Google Drive)

On **Google Colab**: upload the whole project (everything that lives next to `train.py`: `src/`, `configs/`, `requirements.txt`, etc.) into **one folder on Google Drive**. Set `DRIVE_PROJECT_DIR` below to that folder’s path (after mount it looks like `/content/drive/MyDrive/YourFolder`). This cell mounts Drive, `cd`s there, and installs dependencies. CIFAR data and `outputs/` will be created under that same folder unless you change the YAML.

On **local Jupyter**: leave `USE_GOOGLE_DRIVE = False`; the cell uses the current working directory if it already contains `train.py`.

In [7]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Colab + Google Drive: set this to the Drive folder that contains train.py, src/, configs/ ---
USE_GOOGLE_DRIVE = IN_COLAB  # set False on Colab if you prefer git clone /content instead
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/CS4782/cold-diffusion"  # edit to match your Drive folder name

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    root = Path(DRIVE_PROJECT_DIR).resolve()
    if not (root / "train.py").exists():
        raise FileNotFoundError(
            f"train.py not found under {root}. Upload the project into that Drive path "
            "or fix DRIVE_PROJECT_DIR (use My Drive / your folder exactly as in Drive)."
        )
    os.chdir(root)
    %pip install -q -r requirements.txt
elif IN_COLAB:
    # Optional: run from Colab VM without Drive (e.g. after git clone elsewhere)
    local = Path.cwd().resolve()
    if not (local / "train.py").exists():
        raise FileNotFoundError(
            "USE_GOOGLE_DRIVE is False but train.py not in cwd. Clone the repo into /content "
            "or set USE_GOOGLE_DRIVE = True and DRIVE_PROJECT_DIR."
        )
    os.chdir(local)
    %pip install -q -r requirements.txt
else:
    local = Path.cwd().resolve()
    if not (local / "train.py").exists():
        raise FileNotFoundError(
            "train.py not found. Open this notebook from the cold-diffusion repo root, or cd there first."
        )
    os.chdir(local)

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 77.1 MB/s eta 0:00:00
Working directory: /content/drive/MyDrive/Colab Notebooks/CS4782/cold-diffusion


In [11]:
import importlib

# Only run this cell AFTER you have saved changes to your .py files in Google Drive.
# Then you can re-run subsequent cells that depend on these modules.

# Reload train.py and generate.py
import train as train_mod
import generate as generate_mod
import src.degradations.blur as blur_mod
import src.unet as unet_mod
import src.dataset as dataset_mod
import src.degradations.pixelate as pixelate_mod

importlib.reload(pixelate_mod)
importlib.reload(train_mod)
importlib.reload(generate_mod)
importlib.reload(blur_mod)
importlib.reload(unet_mod)
importlib.reload(dataset_mod)

print("Modules reloaded successfully!")

Modules reloaded successfully!


## 2. Imports and device

In [12]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.backends.mps.is_available():
    print("MPS available: True")
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Default tensor device for this check:", device)

torch: 2.10.0+cu128
CUDA available: True
Default tensor device for this check: cuda


## 3. Load config

In [13]:
import yaml

# replace with desired dataset
with open("configs/cifar10.yaml") as f:
    config = yaml.safe_load(f)
config

{'dataset': {'name': 'cifar10',
  'image_size': 32,
  'batch_size': 32,
  'num_workers': 2,
  'root': './data'},
 'model': {'in_channels': 3, 'base_channels': 64, 'time_emb_dim': 256},
 'degradation': {'timesteps': 50},
 'training': {'epochs': 450,
  'learning_rate': 2e-05,
  'output_dir': 'outputs',
  'save_name': 'cold_diffusion_blur.pt'}}

In [ ]:
# config["training"]["epochs"] = 100
# config["dataset"]["batch_size"] = 32
# config["dataset"]["num_workers"] = 2  # Colab-friendly
# config["degradation"]["timesteps"] = 50
pass

## 5. Train (`BlurDegradation`)

In [ ]:
# replace with blur or pixelate depending on which degradation you want to train with
from src.degradations.blur import BlurDegradation as Degradation
import train as train_mod

# downloads dataset into /content/data so that its faster to download and unzip, compared to downloading to Drive
# everything else (checkpoints, outputs) will be saved to Drive, so that they persist across sessions
config['dataset']['root'] = '/content/data'
config['training']['output_dir'] = "outputs/cifar10/blur"

train_mod.Degradation = Degradation
train_mod.train(config)

Initializing training on cuda...
Loading CIFAR10 dataset...


100%|██████████| 170M/170M [00:02<00:00, 72.8MB/s]
Epoch 1/450: 100%|██████████| 1562/1562 [01:53<00:00, 13.74it/s, L1=0.0841, RMSE=0.1284, SSIM=0.7245]


Epoch 1 Completed | L1 Loss: 0.1142 | RMSE: 0.1659 | SSIM: 0.6398



Epoch 2/450: 100%|██████████| 1562/1562 [01:52<00:00, 13.84it/s, L1=0.0778, RMSE=0.1262, SSIM=0.7979]


Epoch 2 Completed | L1 Loss: 0.0847 | RMSE: 0.1307 | SSIM: 0.7567



...

Epoch 160/450: 100%|██████████| 1562/1562 [01:56<00:00, 13.39it/s, L1=0.0426, RMSE=0.0785, SSIM=0.9165]


Epoch 160 Completed | L1 Loss: 0.0449 | RMSE: 0.0827 | SSIM: 0.9007

Checkpoint saved to outputs/cifar10/blur/checkpoint_epoch_160.pt


Epoch 161/450: 100%|██████████| 1562/1562 [01:58<00:00, 13.23it/s, L1=0.0497, RMSE=0.0908, SSIM=0.8803]


Epoch 161 Completed | L1 Loss: 0.0444 | RMSE: 0.0821 | SSIM: 0.9020



Epoch 162/450:   4%|▍         | 65/1562 [00:04<01:42, 14.62it/s, L1=0.0414, RMSE=0.0743, SSIM=0.9254]

## 6. Generate

In [ ]:
import generate as generate_mod

generate_mod.Degradation = Degradation
generate_mod.generate(config)

## 7. Display sample grids

Paths written by [`generate.py`](generate.py): ground truth, heavily blurred start, final restoration, and a few `x0_hat` frames.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

paths = [
    "samples/mnist/pixelate/ground_truth.png",
    "samples/mnist/pixelate/starting_state_xT.png",
    "samples/mnist/pixelate/generated_grid_final.png",
]
for title, p in zip(
    ["Ground truth", "Starting state x_T", "Final generated grid"], paths
):
    print(title)
    display(Image(filename=p))

x0_dir = Path("samples/mnist/pixelate/x0_predictions")
x0_frames = sorted(x0_dir.glob("x0_hat_step_*.png"))
if x0_frames:
    pick = [x0_frames[0]]
    if len(x0_frames) > 1:
        pick.append(x0_frames[len(x0_frames) // 2])
    if len(x0_frames) > 2:
        pick.append(x0_frames[-1])
    for p in dict.fromkeys(pick):
        print(p.name)
        display(Image(filename=str(p)))
else:
    print("No frames in samples/mnist/pixelate/x0_predictions/")